In [1]:
#import gradio as gr

In [2]:
import os

dir1 = "C:/Users/HEMANG/Desktop/skin/HAM10000_images_part_1"
dir2 = "C:/Users/HEMANG/Desktop/skin/HAM10000_images_part_2"

print("Part 1 images:", len(os.listdir(dir1)))
print("Part 2 images:", len(os.listdir(dir2)))


Part 1 images: 5000
Part 2 images: 5015


In [3]:
import pandas as pd

df = pd.read_csv("C:/Users/HEMANG/Desktop/skin/HAM10000_metadata.csv")
print(df.shape)


(10015, 7)


In [4]:
import glob

# collect image paths
paths1 = glob.glob(dir1 + "/*.jpg")
paths2 = glob.glob(dir2 + "/*.jpg")

all_image_paths = paths1 + paths2

print("Total images found:", len(all_image_paths))


Total images found: 10015


In [5]:
# extract image_ids from file paths
image_id_to_path = {
    os.path.basename(p).replace(".jpg", ""): p
    for p in all_image_paths
}

# keep only metadata rows for available images
df_all = df[df["image_id"].isin(image_id_to_path.keys())].copy()

# add image_path column
df_all["image_path"] = df_all["image_id"].map(image_id_to_path)

print("Metadata matched:", df_all.shape)
df_all["dx"].value_counts()


Metadata matched: (10015, 8)


dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64

In [6]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df_all,
    test_size=0.2,
    random_state=42,
    stratify=df_all["dx"]
)

train_df, val_df = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df["dx"]
)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)


Train: (6409, 8)
Validation: (1603, 8)
Test: (2003, 8)


In [7]:
import numpy as np

# get unique class names
class_names = sorted(train_df["dx"].unique())
class_names


['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']

In [8]:
label_map = {name: idx for idx, name in enumerate(class_names)}
label_map


{'akiec': 0, 'bcc': 1, 'bkl': 2, 'df': 3, 'mel': 4, 'nv': 5, 'vasc': 6}

In [9]:
train_df = train_df.copy()
val_df   = val_df.copy()
test_df  = test_df.copy()

train_df["label"] = train_df["dx"].map(label_map)
val_df["label"]   = val_df["dx"].map(label_map)
test_df["label"]  = test_df["dx"].map(label_map)

train_df.head()


,lesion_id,image_id,dx,dx_type,age,sex,localization,image_path,label
3076,HAM_0006362,ISIC_0030523,nv,follow_up,55.0,male,lower extremity,C:/Users/HEMANG/Desktop/skin/HAM10000_images_p...,5
1085,HAM_0000955,ISIC_0029036,bkl,consensus,70.0,female,trunk,C:/Users/HEMANG/Desktop/skin/HAM10000_images_p...,2
8893,HAM_0002429,ISIC_0028139,nv,histo,35.0,female,hand,C:/Users/HEMANG/Desktop/skin/HAM10000_images_p...,5
8834,HAM_0002552,ISIC_0033232,mel,histo,25.0,male,upper extremity,C:/Users/HEMANG/Desktop/skin/HAM10000_images_p...,4
3035,HAM_0003396,ISIC_0031068,nv,follow_up,45.0,male,lower extremity,C:/Users/HEMANG/Desktop/skin/HAM10000_images_p...,5


In [10]:
import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = img / 255.0
    return img, label

train_ds = tf.data.Dataset.from_tensor_slices(
    (train_df["image_path"], train_df["label"])
).map(load_image, num_parallel_calls=tf.data.AUTOTUNE)\
 .shuffle(1000)\
 .batch(BATCH_SIZE)\
 .prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices(
    (val_df["image_path"], val_df["label"])
).map(load_image, num_parallel_calls=tf.data.AUTOTUNE)\
 .batch(BATCH_SIZE)\
 .prefetch(tf.data.AUTOTUNE)


In [11]:
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Conv2D(32, 3, activation="relu", input_shape=(224,224,3)),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(128, 3, activation="relu"),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(len(class_names), activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()
""

C:\Users\HEMANG\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │    22,151,424 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,246,471 (84.86 MB)

 Trainable params: 22,246,471 (84.86 MB)

 Non-trainable params: 0 (0.00 B)

''

In [12]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

callbacks = [
    ModelCheckpoint(
        r'C:\Users\HEMANG\Desktop\skin\models\model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_accuracy',
        patience=4,
        restore_best_weights=True,
        verbose=1
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=11,        # stop at 11 like before
    callbacks=callbacks
)
print(" Done!")

Epoch 1/11
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6456 - loss: 1.3331
Epoch 1: val_accuracy improved from None to 0.64816, saving model to C:\Users\HEMANG\Desktop\skin\models\model.h5



Epoch 1: finished saving model to C:\Users\HEMANG\Desktop\skin\models\model.h5
201/201 ━━━━━━━━━━━━━━━━━━━━ 267s 1s/step - accuracy: 0.6616 - loss: 1.0853 - val_accuracy: 0.6482 - val_loss: 0.9933
Epoch 2/11
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 916ms/step - accuracy: 0.6727 - loss: 0.9395
Epoch 2: val_accuracy improved from 0.64816 to 0.67249, saving model to C:\Users\HEMANG\Desktop\skin\models\model.h5



Epoch 2: finished saving model to C:\Users\HEMANG\Desktop\skin\models\model.h5
201/201 ━━━━━━━━━━━━━━━━━━━━ 206s 1s/step - accuracy: 0.6720 - loss: 0.9191 - val_accuracy: 0.6725 - val_loss: 0.8531
Epoch 3/11
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 805ms/step - accuracy: 0.6774 - loss: 0.8836
Epoch 3: val_accuracy improved from 0.67249 to 0.67623, saving model to C:\Users\HEMANG\Desktop\skin\models\model.h5



Epoch 3: finished saving model to C:\Users\HEMANG\Desktop\skin\models\model.h5
201/201 ━━━━━━━━━━━━━━━━━━━━ 180s 878ms/step - accuracy: 0.6825 - loss: 0.8709 - val_accuracy: 0.6762 - val_loss: 0.8378
Epoch 4/11
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 791ms/step - accuracy: 0.6981 - loss: 0.8459
Epoch 4: val_accuracy improved from 0.67623 to 0.69744, saving model to C:\Users\HEMANG\Desktop\skin\models\model.h5



Epoch 4: finished saving model to C:\Users\HEMANG\Desktop\skin\models\model.h5
201/201 ━━━━━━━━━━━━━━━━━━━━ 176s 862ms/step - accuracy: 0.6961 - loss: 0.8314 - val_accuracy: 0.6974 - val_loss: 0.8208
Epoch 5/11
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 851ms/step - accuracy: 0.6930 - loss: 0.8210
Epoch 5: val_accuracy did not improve from 0.69744
201/201 ━━━━━━━━━━━━━━━━━━━━ 188s 923ms/step - accuracy: 0.6987 - loss: 0.7984 - val_accuracy: 0.6862 - val_loss: 0.8767
Epoch 6/11
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 789ms/step - accuracy: 0.7127 - loss: 0.7956
Epoch 6: val_accuracy improved from 0.69744 to 0.70243, saving model to C:\Users\HEMANG\Desktop\skin\models\model.h5



Epoch 6: finished saving model to C:\Users\HEMANG\Desktop\skin\models\model.h5
201/201 ━━━━━━━━━━━━━━━━━━━━ 175s 861ms/step - accuracy: 0.7132 - loss: 0.7721 - val_accuracy: 0.7024 - val_loss: 0.7709
Epoch 7/11
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 806ms/step - accuracy: 0.7250 - loss: 0.7608
Epoch 7: val_accuracy improved from 0.70243 to 0.70992, saving model to C:\Users\HEMANG\Desktop\skin\models\model.h5



Epoch 7: finished saving model to C:\Users\HEMANG\Desktop\skin\models\model.h5
201/201 ━━━━━━━━━━━━━━━━━━━━ 179s 878ms/step - accuracy: 0.7246 - loss: 0.7373 - val_accuracy: 0.7099 - val_loss: 0.7654
Epoch 8/11
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 869ms/step - accuracy: 0.7156 - loss: 0.7460
Epoch 8: val_accuracy improved from 0.70992 to 0.71366, saving model to C:\Users\HEMANG\Desktop\skin\models\model.h5



Epoch 8: finished saving model to C:\Users\HEMANG\Desktop\skin\models\model.h5
201/201 ━━━━━━━━━━━━━━━━━━━━ 191s 941ms/step - accuracy: 0.7319 - loss: 0.7154 - val_accuracy: 0.7137 - val_loss: 0.7838
Epoch 9/11
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 792ms/step - accuracy: 0.7449 - loss: 0.6986
Epoch 9: val_accuracy did not improve from 0.71366
201/201 ━━━━━━━━━━━━━━━━━━━━ 175s 860ms/step - accuracy: 0.7430 - loss: 0.6893 - val_accuracy: 0.7006 - val_loss: 0.8443
Epoch 10/11
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 790ms/step - accuracy: 0.7482 - loss: 0.6712
Epoch 10: val_accuracy did not improve from 0.71366
201/201 ━━━━━━━━━━━━━━━━━━━━ 175s 859ms/step - accuracy: 0.7533 - loss: 0.6558 - val_accuracy: 0.7099 - val_loss: 0.7515
Epoch 11/11
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 790ms/step - accuracy: 0.7608 - loss: 0.6166
Epoch 11: val_accuracy improved from 0.71366 to 0.72240, saving model to C:\Users\HEMANG\Desktop\skin\models\model.h5



Epoch 11: finished saving model to C:\Users\HEMANG\Desktop\skin\models\model.h5
201/201 ━━━━━━━━━━━━━━━━━━━━ 176s 863ms/step - accuracy: 0.7730 - loss: 0.5973 - val_accuracy: 0.7224 - val_loss: 0.7645
Restoring model weights from the end of the best epoch: 11.
 Done!


In [13]:
model.save(r"C:\Users\HEMANG\Desktop\skin\models\model.h5")

In [14]:
RUN_TEST_EVAL = False


In [15]:
if RUN_TEST_EVAL:
    import cv2
    import numpy as np

    IMG_SIZE = 224

    X_test, y_test = [], []

    for path, label in zip(test_df["image_path"], test_df["label"]):
        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        img = img / 255.0
        X_test.append(img)
        y_test.append(label)

    X_test = np.array(X_test)
    y_test = np.array(y_test)

    print(X_test.shape, y_test.shape)

    test_loss, test_acc = model.evaluate(X_test, y_test, batch_size=32)
    print("Test Accuracy:", test_acc)
    print("Test Loss:", test_loss)

else:
    print("Skipping test evaluation (already completed)")
    print("Saved Result → Test Accuracy ≈ 71.1%")





Skipping test evaluation (already completed)
Saved Result → Test Accuracy ≈ 71.1%


In [16]:
from tensorflow.keras.models import load_model

model = load_model(r"C:\Users\HEMANG\Desktop\skin\models\model.h5")
print("CNN model loaded")


CNN model loaded


In [17]:
import tensorflow as tf
from tensorflow.keras.models import Model

# recreate input
inputs = tf.keras.Input(shape=(224, 224, 3))

x = inputs
output_found = False
for layer in model.layers:
    x = layer(x)
    # Identify the dense layer with 256 units by its output shape
    if isinstance(layer, tf.keras.layers.Dense) and layer.units == 256:
        output_found = True
        break # Stop after the 256-unit dense layer

if not output_found:
    print("Warning: Could not find a Dense layer with 256 units for feature extraction.")

feature_extractor = Model(inputs=inputs, outputs=x)

print("Feature extractor ready (256-d features)")

Feature extractor ready (256-d features)


In [18]:
import numpy as np

dummy = np.zeros((1, 224, 224, 3))
out = feature_extractor.predict(dummy)

print(out.shape)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 259ms/step
(1, 256)


In [19]:
def extract_features(df):
    features = []
    labels = []

    for path, label in zip(df["image_path"], df["label"]):
        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (224, 224))
        img = img / 255.0

        feat = feature_extractor.predict(
            np.expand_dims(img, axis=0),
            verbose=0
        )[0]

        features.append(feat)
        labels.append(label)

    return np.array(features), np.array(labels)


In [20]:
import cv2

In [21]:
X_train_feat, y_train = extract_features(train_df)
X_test_feat, y_test = extract_features(test_df)

print(X_train_feat.shape, X_test_feat.shape)


(6409, 256) (2003, 256)


In [22]:
import os
os.makedirs(r"C:\Users\HEMANG\Desktop\skin\data", exist_ok=True)

np.save(r"C:\Users\HEMANG\Desktop\skin\data\X_train_feat.npy", X_train_feat)
np.save(r"C:\Users\HEMANG\Desktop\skin\data\y_train.npy", y_train)
np.save(r"C:\Users\HEMANG\Desktop\skin\data\X_test_feat.npy", X_test_feat)
np.save(r"C:\Users\HEMANG\Desktop\skin\data\y_test.npy", y_test)
print("Features saved successfully ")

Features saved successfully 


In [23]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

svm = SVC(kernel="rbf", C=10, gamma="scale", class_weight="balanced")

svm.fit(X_train_feat, y_train)

y_pred_svm = svm.predict(X_test_feat)

svm_acc = accuracy_score(y_test, y_pred_svm)
print("SVM Accuracy:", svm_acc)

print(classification_report(y_test, y_pred_svm))

SVM Accuracy: 0.6515227159261109
              precision    recall  f1-score   support

           0       0.27      0.37      0.31        65
           1       0.37      0.49      0.42       103
           2       0.37      0.50      0.43       220
           3       0.15      0.26      0.19        23
           4       0.35      0.55      0.43       223
           5       0.92      0.73      0.81      1341
           6       0.58      0.64      0.61        28

    accuracy                           0.65      2003
   macro avg       0.43      0.51      0.46      2003
weighted avg       0.73      0.65      0.68      2003



In [24]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_feat, y_train)

y_pred_rf = rf.predict(X_test_feat)

rf_acc = accuracy_score(y_test, y_pred_rf)
print("Random Forest Accuracy:", rf_acc)

Random Forest Accuracy: 0.7358961557663505


In [25]:
import pandas as pd

results = pd.DataFrame({
    "Model": ["CNN", "SVM (CNN features)", "Random Forest (CNN features)"],
    "Accuracy": [
        0.7224,        #  CNN test accuracy
        svm_acc,
        rf_acc
    ]
})

results


,Model,Accuracy
0,CNN,0.722400
1,SVM (CNN features),0.651523
2,Random Forest (CNN features),0.735896


In [26]:
import joblib, os

os.makedirs(r"C:\Users\HEMANG\Desktop\skin\models", exist_ok=True)

joblib.dump(svm, r"C:\Users\HEMANG\Desktop\skin\models\svm_best.pkl")
joblib.dump(rf,  r"C:\Users\HEMANG\Desktop\skin\models\rf_best.pkl")
model.save(r"C:\Users\HEMANG\Desktop\skin\models\model.h5")

print("All models saved ")


All models saved 


In [27]:
import os
import joblib

# Create models folder
os.makedirs('C:/Skin/models', exist_ok=True)

# First check variable names
print("Available variables:", [x for x in dir() if any(k in x.lower() for k in ['model','svm','clf'])])

Available variables: ['Model', 'ModelCheckpoint', 'load_model', 'model', 'models', 'svm', 'svm_acc', 'y_pred_svm']


In [28]:
# Save CNN model
model.save('C:/Skin/models/model.h5')
print(" CNN model saved!")

# Save SVM model  
joblib.dump(svm, r'C:\Users\HEMANG\Desktop\skin\models\svm_model.pkl')
print(" SVM model saved!")

 CNN model saved!
 SVM model saved!
